In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/Gland Stock Price History.csv')

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

# --- Acknowledgment of dataset discrepancy ---
# The user specified 'ICICIBANK.NS.csv' but the current DataFrame 'df' is loaded from 'Gland Stock Price History.csv'.
# I will proceed with the 'Gland Stock Price History.csv' data as it is available in the kernel state.
# Please ensure 'ICICIBANK.NS.csv' is uploaded and 'df = pd.read_csv('ICICIBANK.NS.csv')' is executed if you wish to use that data.

# --- STEP 1: DATA CLEANING & PREPROCESSING ---

print("--- Starting Data Cleaning & Preprocessing ---")

# Ensure 'Date' is a column before proceeding with date conversion and setting as index.
# If 'Date' is already the index, reset it to make it a column.
if df.index.name == 'Date':
    df.reset_index(inplace=True)
elif df.index.name is not None: # If some other column is index, reset it too
    df.reset_index(inplace=True)

# 1. Column Renaming to match typical financial data conventions
# Only rename if the column actually exists to avoid KeyError if it was already renamed or doesn't exist.
rename_map = {}
if 'Price' in df.columns:
    rename_map['Price'] = 'Close'
if 'Vol.' in df.columns:
    rename_map['Vol.'] = 'Volume'
if 'Change %' in df.columns:
    rename_map['Change %'] = 'Daily_Percentage_Change'

if rename_map:
    df.rename(columns=rename_map, inplace=True)

# 2. Datetime conversion and Sorting
# Now 'Date' should be a column.
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y', errors='coerce')
df.sort_values(by='Date', inplace=True)

# 3. Set 'Date' as index
df.set_index('Date', inplace=True)

# 4. Clean numeric columns (Open, High, Low, Close)
# Apply a function to remove commas and convert to numeric for relevant columns
def clean_and_convert_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(',', '', regex=False), errors='coerce')

for col in ['Open', 'High', 'Low', 'Close']:
    if col in df.columns:
        df[col] = clean_and_convert_numeric(df[col])

# 5. Clean 'Volume' column
# Convert 'Volume' to numeric, handling 'M' for millions and 'K' for thousands.
def clean_volume(volume_str):
    if isinstance(volume_str, str):
        volume_str = volume_str.strip()
        if 'M' in volume_str:
            return float(volume_str.replace('M', '')) * 1_000_000
        elif 'K' in volume_str:
            return float(volume_str.replace('K', '')) * 1_000
        else:
            try:
                return float(volume_str.replace(',', '')) # Handle commas in numbers
            except ValueError:
                return np.nan # Return NaN for unparseable strings
    return volume_str # Return as is if not a string (e.g., already a number or NaN)

# Ensure 'Volume' is processed as strings before applying clean_volume
if 'Volume' in df.columns:
    df['Volume'] = df['Volume'].astype(str).apply(clean_volume)
    df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce') # Ensure numeric, coerce errors to NaN

# 6. Clean 'Daily_Percentage_Change' column
# Ensure it's string, remove '%' and convert to float
# Replace empty strings resulting from cleaning with NaN before converting to float
if 'Daily_Percentage_Change' in df.columns:
    df['Daily_Percentage_Change'] = df['Daily_Percentage_Change'].astype(str).str.replace('%', '', regex=False).replace('', np.nan).astype(float) / 100
    # Handle potential NaN values after conversion if any remain or if parsing failed
    df['Daily_Percentage_Change'].replace([np.inf, -np.inf], np.nan, inplace=True)

# 7. Missing value handling
print(f"Missing values before handling:\n{df.isnull().sum()}")
# For financial time series, forward-fill is often used for missing data,
# but for critical columns like 'Open', 'High', 'Low', 'Close', 'Volume',
# dropping rows with NaNs is safer to maintain data integrity.
# For 'Daily_Returns', 'Volatility', 'MA', 'Cumulative_Returns', NaNs will naturally appear
# due to rolling window calculations at the start of the series.
critical_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
# Add 'Daily_Percentage_Change' if it exists in df.columns after potential rename
if 'Daily_Percentage_Change' in df.columns:
    critical_columns.append('Daily_Percentage_Change')

# Filter critical_columns to only include those actually present in df
critical_columns_present = [col for col in critical_columns if col in df.columns]

if critical_columns_present:
    df.dropna(subset=critical_columns_present, inplace=True)
print(f"Missing values after dropping critical NaNs:\n{df.isnull().sum()}")

# 8. Duplicate removal
initial_rows = len(df)
df.drop_duplicates(inplace=True)
if len(df) < initial_rows:
    print(f"Removed {initial_rows - len(df)} duplicate rows.")
else:
    print("No duplicate rows found.")

# 9. Outlier check (basic description)
# A full outlier detection and treatment would involve statistical methods or domain knowledge.
# For this step, we'll provide descriptive statistics to quickly inspect for anomalies.
print("\nDescriptive statistics after cleaning:")
describe_cols = [col for col in ['Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Percentage_Change'] if col in df.columns]
if describe_cols:
    print(df[describe_cols].describe())
else:
    print("No numerical columns to describe after cleaning.")
# Further outlier detection could involve Z-scores, IQR, or visual inspection (box plots).

print("\n--- Creating Financial Indicators ---")

# Create: Daily Returns
# Ensure 'Close' is numeric before pct_change
if 'Close' in df.columns:
    df['Daily_Returns'] = df['Close'].pct_change()
else:
    df['Daily_Returns'] = np.nan # Or handle appropriately if 'Close' is missing

# Create: Volatility (Annualized 20-day rolling standard deviation of daily returns)
# Assuming 252 trading days in a year
if 'Daily_Returns' in df.columns:
    df['Volatility'] = df['Daily_Returns'].rolling(window=20).std() * np.sqrt(252)
else:
    df['Volatility'] = np.nan

# Create: Moving Averages (20, 50, 100 days)
if 'Close' in df.columns:
    df['MA_20'] = df['Close'].rolling(window=20).mean()
    df['MA_50'] = df['Close'].rolling(window=50).mean()
    df['MA_100'] = df['Close'].rolling(window=100).mean()
else:
    df['MA_20'] = np.nan
    df['MA_50'] = np.nan
    df['MA_100'] = np.nan

# Create: Cumulative Returns
if 'Daily_Returns' in df.columns:
    df['Cumulative_Returns'] = (1 + df['Daily_Returns']).cumprod() - 1
else:
    df['Cumulative_Returns'] = np.nan

print("\n--- Data Cleaning & Preprocessing Complete ---")
print("First 5 rows of the processed DataFrame:")
print(df.head())
print("\nLast 5 rows of the processed DataFrame:")
print(df.tail())
print(f"\nDataFrame shape after preprocessing: {df.shape}")

--- Starting Data Cleaning & Preprocessing ---
Missing values before handling:
Close                      0
Open                       0
High                       0
Low                        0
Volume                     0
Daily_Percentage_Change    0
dtype: int64
Missing values after dropping critical NaNs:
Close                      0
Open                       0
High                       0
Low                        0
Volume                     0
Daily_Percentage_Change    0
dtype: int64
No duplicate rows found.

Descriptive statistics after cleaning:
              Open         High          Low        Close        Volume  \
count  1358.000000  1358.000000  1358.000000  1358.000000  1.358000e+03   
mean   2149.243409  2184.711856  2109.970398  2145.096649  3.908285e+05   
std     767.335073   778.958425   750.806444   765.323282  8.274763e+05   
min     908.500000   920.000000   861.000000   893.550000  7.500000e+03   
25%    1674.225000  1697.687500  1647.075000  1669.900000  1.1

/tmp/ipykernel_1610/3594471118.py:85: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Daily_Percentage_Change'].replace([np.inf, -np.inf], np.nan, inplace=True)


In [ ]:
print("\n--- STEP 2: KPI METRICS ---")

# Calculate required metrics
latest_closing_price = df['Close'].iloc[-1]
highest_price = df['High'].max()
lowest_price = df['Low'].min()
average_volume = df['Volume'].mean()

# Total Return %
# Using the first and last non-NaN Close price for total return calculation
initial_close = df['Close'].iloc[0]
final_close = df['Close'].iloc[-1]
total_return_percent = ((final_close - initial_close) / initial_close) * 100

# CAGR (Compound Annual Growth Rate)
# Number of years = (last_date - first_date).days / 365.25
first_date = df.index.min()
last_date = df.index.max()
num_years = (last_date - first_date).days / 365.25
cagr = ((final_close / initial_close)**(1 / num_years) - 1) * 100

# Volatility % (Annualized Standard Deviation of Daily Returns)
# We already calculated a rolling volatility. For a single KPI, we'll use the overall std of daily returns.
annualized_volatility_percent = df['Daily_Returns'].std() * np.sqrt(252) * 100

# Sharpe Ratio (Requires Risk-Free Rate, assume 0 for simplicity or specify)
# Assuming 0 risk-free rate for simplicity
risk_free_rate = 0.0 # Placeholder, replace with actual risk-free rate if available
excess_returns = df['Daily_Returns'] - (risk_free_rate / 252)
sharpe_ratio = np.mean(excess_returns) / np.std(excess_returns) * np.sqrt(252)

# Max Drawdown
# Calculate the cumulative return series (wealth index)
wealth_index = (1 + df['Daily_Returns']).cumprod()
previous_peaks = wealth_index.cummax()
drawdown = (wealth_index - previous_peaks) / previous_peaks
max_drawdown = drawdown.min() * 100

print("\n--- Key Performance Indicators (KPIs) ---")
print(f"Latest Closing Price: {latest_closing_price:,.2f}")
print(f"Highest Price (overall): {highest_price:,.2f}")
print(f"Lowest Price (overall): {lowest_price:,.2f}")
print(f"Average Volume: {average_volume:,.0f}")
print(f"Total Return %: {total_return_percent:.2f}%")
print(f"CAGR (Compound Annual Growth Rate): {cagr:.2f}%")
print(f"Annualized Volatility %: {annualized_volatility_percent:.2f}%")
print(f"Sharpe Ratio (assuming 0% risk-free rate): {sharpe_ratio:.2f}")
print(f"Maximum Drawdown %: {max_drawdown:.2f}%")

print("\n--- KPI Metrics Calculation Complete ---")


--- STEP 2: KPI METRICS ---

--- Key Performance Indicators (KPIs) ---
Latest Closing Price: 2,251.10
Highest Price (overall): 4,350.00
Lowest Price (overall): 861.00
Average Volume: 390,829
Total Return %: 7.76%
CAGR (Compound Annual Growth Rate): 1.37%
Annualized Volatility %: 37.89%
Sharpe Ratio (assuming 0% risk-free rate): 0.23
Maximum Drawdown %: -79.30%

--- KPI Metrics Calculation Complete ---


In [ ]:
print("\n--- STEP 3: DATA VISUALIZATION ---")

# Set a finance-themed template for Plotly
import plotly.io as pio
pio.templates.default = "plotly_dark" # or 'plotly_white', 'seaborn', 'ggplot2', etc.

# 1. Closing Price Trend
print("Generating Closing Price Trend...")
fig = px.line(df, x=df.index, y='Close', title='Closing Price Trend for ICICI Bank')
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Closing Price",
    hovermode="x unified",
    template="plotly_dark"
)
fig.show()

# 2. Open vs Close Comparison
print("Generating Open vs Close Comparison...")
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['Open'], mode='lines', name='Open Price', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Close Price', line=dict(color='lightblue')))
fig.update_layout(
    title_text='Open vs Close Price Comparison for ICICI Bank',
    xaxis_title="Date",
    yaxis_title="Price",
    hovermode="x unified",
    template="plotly_dark"
)
fig.show()

# 3. High vs Low Chart
print("Generating High vs Low Chart...")
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['High'], mode='lines', name='High Price', line=dict(color='lightgreen')))
fig.add_trace(go.Scatter(x=df.index, y=df['Low'], mode='lines', name='Low Price', line=dict(color='red')))
fig.update_layout(
    title_text='High vs Low Price Range for ICICI Bank',
    xaxis_title="Date",
    yaxis_title="Price",
    hovermode="x unified",
    template="plotly_dark"
)
fig.show()

# 4. Candlestick Chart
print("Generating Candlestick Chart...")
fig = go.Figure(data=[
    go.Candlestick(
        x=df.index,
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        name='Candlestick'
    )
])
fig.update_layout(
    title_text='Candlestick Chart for ICICI Bank',
    xaxis_title="Date",
    yaxis_title="Price",
    hovermode="x unified",
    xaxis_rangeslider_visible=False, # Hide range slider for cleaner look
    template="plotly_dark"
)
fig.show()

# 5. Trading Volume Bar Chart
print("Generating Trading Volume Bar Chart...")
fig = px.bar(df, x=df.index, y='Volume', title='Trading Volume for ICICI Bank', color_discrete_sequence=['purple'])
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Volume",
    hovermode="x unified",
    template="plotly_dark"
)
fig.show()

print("--- Data Visualization (Part 1) Complete ---")


--- STEP 3: DATA VISUALIZATION ---
Generating Closing Price Trend...


Generating Open vs Close Comparison...


Generating High vs Low Chart...


Generating Candlestick Chart...


Generating Trading Volume Bar Chart...


--- Data Visualization (Part 1) Complete ---


In [ ]:
print("--- Data Visualization (Part 2) ---")

# 6. Daily Returns Histogram
print("Generating Daily Returns Histogram...")
fig = px.histogram(df, x='Daily_Returns', nbins=50, title='Distribution of Daily Returns', color_discrete_sequence=['gold'])
fig.update_layout(
    xaxis_title="Daily Returns",
    yaxis_title="Frequency",
    template="plotly_dark"
)
fig.show()

# 7. Volatility Trend
print("Generating Volatility Trend...")
fig = px.line(df, x=df.index, y='Volatility', title='Annualized Rolling 20-Day Volatility Trend', color_discrete_sequence=['orange'])
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Annualized Volatility",
    hovermode="x unified",
    template="plotly_dark"
)
fig.show()

# 8. Rolling Mean Chart (20, 50, 100-Day Moving Averages)
# Combining 9, 10, 11 into one for better comparison
print("Generating Rolling Mean Chart (20, 50, 100-Day MA)...")
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Close Price', line=dict(color='lightblue')))
fig.add_trace(go.Scatter(x=df.index, y=df['MA_20'], mode='lines', name='20-Day MA', line=dict(color='green', dash='dot')))
fig.add_trace(go.Scatter(x=df.index, y=df['MA_50'], mode='lines', name='50-Day MA', line=dict(color='red', dash='dash')))
fig.add_trace(go.Scatter(x=df.index, y=df['MA_100'], mode='lines', name='100-Day MA', line=dict(color='purple', dash='longdash')))
fig.update_layout(
    title_text='Closing Price with 20, 50, and 100-Day Moving Averages',
    xaxis_title="Date",
    yaxis_title="Price",
    hovermode="x unified",
    template="plotly_dark"
)
fig.show()

# 12. Correlation Heatmap
print("Generating Correlation Heatmap...")
# Select relevant numerical columns for correlation
correlation_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Returns', 'Volatility']
corr_matrix = df[correlation_cols].corr()
fig = px.imshow(corr_matrix, text_auto=True, aspect="auto", title='Correlation Heatmap of Financial Metrics')
fig.update_layout(
    template="plotly_dark"
)
fig.show()

# 13. Monthly Performance Heatmap
print("Generating Monthly Performance Heatmap...")
# Ensure 'Daily_Returns' is not NaN for this calculation
monthly_returns = df['Daily_Returns'].resample('M').apply(lambda x: (1 + x).prod() - 1)
monthly_returns_df = pd.DataFrame({
    'Year': monthly_returns.index.year,
    'Month': monthly_returns.index.month_name(),
    'Returns': monthly_returns
})
# Pivot the table for heatmap
monthly_performance = monthly_returns_df.pivot_table(index='Month', columns='Year', values='Returns')
# Order months correctly
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
monthly_performance = monthly_performance.reindex(month_order)
fig = px.imshow(monthly_performance, text_auto=True, aspect="auto",
                title='Monthly Performance Heatmap',
                color_continuous_scale='RdYlGn', # Red-Yellow-Green for performance
                labels=dict(x="Year", y="Month", color="Return"))
fig.update_layout(
    template="plotly_dark"
)
fig.show()

# 14. Yearly Return Comparison
print("Generating Yearly Return Comparison...")
yearly_returns = df['Daily_Returns'].resample('Y').apply(lambda x: (1 + x).prod() - 1)
yearly_returns = yearly_returns.to_frame(name='Returns')
yearly_returns['Year'] = yearly_returns.index.year
fig = px.bar(yearly_returns, x='Year', y='Returns', title='Yearly Return Comparison', color='Returns',
             color_continuous_scale='RdYlGn')
fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Return",
    template="plotly_dark"
)
fig.show()

# 15. Boxplot of Daily Returns
print("Generating Boxplot of Daily Returns...")
fig = px.box(df, y='Daily_Returns', title='Boxplot of Daily Returns Distribution', color_discrete_sequence=['teal'])
fig.update_layout(
    yaxis_title="Daily Returns",
    template="plotly_dark"
)
fig.show()

# 16. Cumulative Return Graph
print("Generating Cumulative Return Graph...")
fig = px.line(df, x=df.index, y='Cumulative_Returns', title='Cumulative Returns Over Time', color_discrete_sequence=['cyan'])
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Cumulative Returns",
    hovermode="x unified",
    template="plotly_dark"
)
fig.show()

# 17. Seasonal Decomposition Plot (for 'Close' prices)
print("Generating Seasonal Decomposition Plot...")
# Resample to weekly or monthly to avoid too many points for decomposition if daily data is very long
# For simplicity, let's resample to monthly close prices.
# We need a frequency for seasonal_decompose, infer from data if possible.
# If data is daily and has many years, a yearly seasonality might be appropriate.
# Let's try to decompose on a monthly average if date range is sufficient, or raw close if not.

# Check if the data length and frequency are suitable for seasonal decomposition
if len(df) > 2 * 12: # At least two years for monthly seasonality
    # Use monthly average 'Close' prices for decomposition
    monthly_close = df['Close'].resample('M').mean()
    # Try to infer frequency, assume 12 for monthly data (yearly seasonality)
    try:
        decomposition = seasonal_decompose(monthly_close.dropna(), model='additive', period=12)

        fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                            subplot_titles=('Original Series', 'Trend Component', 'Seasonal Component', 'Residual Component'))

        fig.add_trace(go.Scatter(x=decomposition.observed.index, y=decomposition.observed, mode='lines', name='Original'), row=1, col=1)
        fig.add_trace(go.Scatter(x=decomposition.trend.index, y=decomposition.trend, mode='lines', name='Trend'), row=2, col=1)
        fig.add_trace(go.Scatter(x=decomposition.seasonal.index, y=decomposition.seasonal, mode='lines', name='Seasonal'), row=3, col=1)
        fig.add_trace(go.Scatter(x=decomposition.resid.index, y=decomposition.resid, mode='lines', name='Residual'), row=4, col=1)

        fig.update_layout(
            title_text='Seasonal Decomposition of Monthly Close Price',
            hovermode="x unified",
            height=900,
            template="plotly_dark"
        )
        fig.show()
    except Exception as e:
        print(f"Could not perform seasonal decomposition: {e}. Data might be too short or irregular.")
        print("Consider checking data frequency or length.")
else:
    print("Not enough data points for meaningful seasonal decomposition. Requires at least 2 full cycles (e.g., 24 months for monthly seasonality).")


print("--- Data Visualization (Part 2) Complete ---")

--- Data Visualization (Part 2) ---
Generating Daily Returns Histogram...


Generating Volatility Trend...


Generating Rolling Mean Chart (20, 50, 100-Day MA)...


Generating Correlation Heatmap...


Generating Monthly Performance Heatmap...


...

Generating Boxplot of Daily Returns...


Generating Cumulative Return Graph...


Generating Seasonal Decomposition Plot...


/tmp/ipykernel_1610/1465007901.py:120: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



--- Data Visualization (Part 2) Complete ---


In [ ]:
print("\n--- STEP 4: TIME SERIES ANALYSIS ---")

# ARIMA Forecasting
print("Performing ARIMA Forecasting...")

# For ARIMA, we typically use the 'Close' price and ensure it's stationary.
# Check for stationarity using Augmented Dickey-Fuller Test
# (skipped here for brevity, assuming differencing will make it stationary)

# Differencing the 'Close' price to make it stationary
# We'll use the first difference, as stock prices are often non-stationary.
df['Close_diff'] = df['Close'].diff().dropna()

# Select the data for ARIMA model
# We need to drop NaNs created by differencing
arima_data = df['Close_diff'].dropna()

# Determine p, d, q parameters for ARIMA (using auto_arima or manual ACF/PACF analysis is common)
# For this example, let's choose some common parameters. d=1 because we differenced once.
# p and q can be found using ACF/PACF plots or auto_arima.
# Let's start with a simple ARIMA(5,1,0) - AR(5), I(1), MA(0)
# A more robust approach would involve iterating through p,d,q values and AIC/BIC scores.

# Splitting data into train and test sets (optional, but good practice for forecasting evaluation)
train_size = int(len(arima_data) * 0.8)
train_data, test_data = arima_data[0:train_size], arima_data[train_size:]

# Fit ARIMA model
# Using the non-differenced 'Close' series and letting ARIMA handle differencing (d=1)
# This simplifies the prediction inversion step.
model = ARIMA(df['Close'].dropna(), order=(5,1,0)) # ARIMA(p,d,q) where d=1 for first differencing
model_fit = model.fit()

print(model_fit.summary())

# 18. ARIMA Forecast Chart & 30-day future prediction
print("Generating 30-day future prediction and ARIMA Forecast Chart...")

# Forecast for the next 30 days
# Use 'steps' for the number of periods to forecast
forecast_steps = 30
forecast = model_fit.forecast(steps=forecast_steps)
forecast_index = pd.date_range(start=df.index.max() + timedelta(days=1), periods=forecast_steps, freq='B') # 'B' for business day frequency
forecast_series = pd.Series(forecast.values, index=forecast_index)

# Plot the forecast
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Actual Close Price', line=dict(color='lightblue')))
fig.add_trace(go.Scatter(x=forecast_series.index, y=forecast_series.values, mode='lines', name='ARIMA Forecast', line=dict(color='orange', dash='dash')))

fig.update_layout(
    title_text=f'ARIMA Forecast for Next {forecast_steps} Business Days',
    xaxis_title="Date",
    yaxis_title="Close Price",
    hovermode="x unified",
    template="plotly_dark"
)
fig.show()

# Trend Interpretation
print("\n--- Trend Interpretation from ARIMA Model ---")
# Interpretation of the trend component from seasonal decomposition was done visually.
# For ARIMA, trend is captured by the differencing (d) and AR components.
# A positive forecast suggests an upward trend, negative suggests downward.
if forecast_series.iloc[-1] > df['Close'].iloc[-1]:
    trend_interpretation = "The ARIMA model forecasts an upward trend in the closing price over the next 30 business days."
else:
    trend_interpretation = "The ARIMA model forecasts a downward trend or stabilization in the closing price over the next 30 business days."

print(trend_interpretation)

print("\n--- Time Series Analysis Complete ---")


--- STEP 4: TIME SERIES ANALYSIS ---
Performing ARIMA Forecasting...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.



                               SARIMAX Results                                
Dep. Variable:                  Close   No. Observations:                 1358
Model:                 ARIMA(5, 1, 0)   Log Likelihood               -7280.912
Date:                Tue, 19 May 2026   AIC                          14573.824
Time:                        14:06:42   BIC                          14605.102
Sample:                             0   HQIC                         14585.535
                               - 1358                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.0206      0.017     -1.241      0.215      -0.053       0.012
ar.L2         -0.0815      0.022     -3.703      0.000      -0.125      -0.038
ar.L3         -0.0226      0.021     -1.050      0.2

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.




--- Trend Interpretation from ARIMA Model ---
The ARIMA model forecasts a downward trend or stabilization in the closing price over the next 30 business days.

--- Time Series Analysis Complete ---


In [ ]:
print("\n--- STEP 5: ALGORITHMIC TRADING ---")

# Create a DataFrame for trading signals
signals = pd.DataFrame(index=df.index)
signals['Close'] = df['Close']
signals['MA_20'] = df['MA_20']
signals['MA_50'] = df['MA_50']

# Generate Buy/Sell signals
# Rule: Buy when 20-Day MA crosses ABOVE 50-Day MA
# Rule: Sell when 20-Day MA crosses BELOW 50-Day MA

signals['Signal'] = 0.0
signals['Position'] = 0.0 # 1 for long position, -1 for short, 0 for no position

# Create a 'signal' column where 1 is a buy signal and -1 is a sell signal
signals['Signal'][signals['MA_20'] > signals['MA_50']] = 1
signals['Signal'][signals['MA_20'] < signals['MA_50']] = -1

# Calculate position changes
signals['Position'] = signals['Signal'].diff()

# Remove NaN values from the beginning of the signals DataFrame
signals.dropna(inplace=True)

print("\n--- Trading Signal Generation Complete ---")
print("First 5 rows of Trading Signals:")
print(signals.head())
print("Last 5 rows of Trading Signals:")
print(signals.tail())

# 19. Buy/Sell Signal Visualization
print("Generating Buy/Sell Signal Visualization...")

fig = go.Figure()

# Plot Close Price
fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Close Price', line=dict(color='lightblue')))

# Plot Moving Averages
fig.add_trace(go.Scatter(x=df.index, y=df['MA_20'], mode='lines', name='20-Day MA', line=dict(color='green')))
fig.add_trace(go.Scatter(x=df.index, y=df['MA_50'], mode='lines', name='50-Day MA', line=dict(color='red')))

# Plot Buy Signals (20-Day MA crosses above 50-Day MA)
# We'll plot buy signals where 'Position' changes from -1 to 1 or 0 to 1
buy_signals = signals[signals['Position'] == 1]
fig.add_trace(go.Scatter(
    x=buy_signals.index,
    y=df['Close'].loc[buy_signals.index],
    mode='markers',
    marker=dict(symbol='triangle-up', size=10, color='green', line=dict(width=1, color='DarkSlateGrey')),
    name='Buy Signal'
))

# Plot Sell Signals (20-Day MA crosses below 50-Day MA)
# We'll plot sell signals where 'Position' changes from 1 to -1 or 0 to -1
sell_signals = signals[signals['Position'] == -1]
fig.add_trace(go.Scatter(
    x=sell_signals.index,
    y=df['Close'].loc[sell_signals.index],
    mode='markers',
    marker=dict(symbol='triangle-down', size=10, color='red', line=dict(width=1, color='DarkSlateGrey')),
    name='Sell Signal'
))

fig.update_layout(
    title_text='Moving Average Crossover Trading Signals',
    xaxis_title="Date",
    yaxis_title="Price",
    hovermode="x unified",
    template="plotly_dark"
)
fig.show()

# Signal Table (Display a summary of trades)
print("\n--- Trading Signal Table ---")
trade_log = []
is_in_position = False
buy_price = 0

for i in range(1, len(signals.index)):
    current_date = signals.index[i]
    previous_signal = signals['Signal'].iloc[i-1]
    current_signal = signals['Signal'].iloc[i]

    if current_signal == 1 and previous_signal != 1 and not is_in_position: # Buy signal (MA_20 > MA_50)
        buy_price = df['Close'].loc[current_date]
        trade_log.append({'Type': 'BUY', 'Date': current_date, 'Price': buy_price})
        is_in_position = True
    elif current_signal == -1 and previous_signal != -1 and is_in_position: # Sell signal (MA_20 < MA_50)
        sell_price = df['Close'].loc[current_date]
        trade_log.append({'Type': 'SELL', 'Date': current_date, 'Price': sell_price})
        is_in_position = False

trade_df = pd.DataFrame(trade_log)
print(trade_df.to_string())

print("\n--- Algorithmic Trading Step Complete ---")


--- STEP 5: ALGORITHMIC TRADING ---

--- Trading Signal Generation Complete ---
First 5 rows of Trading Signals:
              Close      MA_20     MA_50  Signal  Position
Date                                                      
2021-02-03  2132.10  2248.8050  2254.467    -1.0      -1.0
2021-02-04  2136.85  2235.2250  2255.423    -1.0       0.0
2021-02-05  2293.70  2229.8550  2260.327    -1.0       0.0
2021-02-08  2322.55  2227.5775  2265.718    -1.0       0.0
2021-02-09  2268.30  2224.5575  2268.950    -1.0       0.0
Last 5 rows of Trading Signals:
             Close     MA_20     MA_50  Signal  Position
Date                                                    
2026-05-13  1844.2  1800.385  1740.656     1.0       0.0
2026-05-14  1898.2  1807.660  1741.650     1.0       0.0
2026-05-15  1868.3  1814.295  1741.792     1.0       0.0
2026-05-18  2156.6  1832.255  1748.480     1.0       0.0
2026-05-19  2251.1  1855.080  1757.606     1.0       0.0
Generating Buy/Sell Signal Visualization..

/tmp/ipykernel_1610/3966840295.py:17: FutureWarning:

ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


/tmp/ipykernel_1610/3966840295.py:18: FutureWarning:

ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained ass


--- Trading Signal Table ---
    Type       Date    Price
0    BUY 2021-03-02  2349.85
1   SELL 2021-09-17  3962.90
2    BUY 2021-12-24  3893.85
3   SELL 2022-02-01  3512.95
4    BUY 2022-09-12  2361.40
5   SELL 2022-09-27  2060.45
6    BUY 2023-04-21  1311.20
7   SELL 2023-05-26   929.25
8    BUY 2023-07-18  1161.20
9   SELL 2023-10-20  1558.90
10   BUY 2023-11-28  1785.10
11  SELL 2024-03-01  1787.55
12   BUY 2024-05-30  1850.20
13  SELL 2024-09-02  1832.95
14   BUY 2024-11-27  1728.40
15  SELL 2025-01-22  1633.55
16   BUY 2025-03-20  1591.65
17  SELL 2025-04-16  1442.60
18   BUY 2025-06-02  1611.80
19  SELL 2025-09-04  1886.00
20   BUY 2025-10-01  1981.60
21  SELL 2025-10-24  1941.40
22   BUY 2026-02-01  1834.30
23  SELL 2026-03-13  1626.60
24   BUY 2026-04-28  1760.80

--- Algorithmic Trading Step Complete ---


In [ ]:
print("\n--- STEP 6: BACKTESTING ---")

# --- Backtesting Performance Calculation ---

# Calculate daily strategy returns
# Initialize strategy returns to 0
signals['Strategy_Returns'] = 0.0

# Set strategy returns to daily returns only when in position (Signal = 1 or -1)
# We will simplify by assuming we are always either long (1) or short (-1) based on the signal.
# A more complex strategy would involve holding cash when signal is 0 or only going long.
# For this crossover, let's assume we are always in the market (long or short based on signal).
# Re-calculate Position to reflect holding from signal generation
signals_with_position = pd.DataFrame(index=df.index)
signals_with_position['Close'] = df['Close']
signals_with_position['MA_20'] = df['MA_20']
signals_with_position['MA_50'] = df['MA_50']
signals_with_position['Signal'] = 0.0
signals_with_position['Signal'][signals_with_position['MA_20'] > signals_with_position['MA_50']] = 1
signals_with_position['Signal'][signals_with_position['MA_20'] < signals_with_position['MA_50']] = -1
signals_with_position['Signal'] = signals_with_position['Signal'].ffill().fillna(0) # Forward fill signals to maintain position

signals_with_position['Market_Returns'] = df['Daily_Returns']
signals_with_position['Strategy_Returns'] = signals_with_position['Market_Returns'] * signals_with_position['Signal'].shift(1)

# Drop NaN values (from initial MA calculations and first shifted signal)
signals_with_position.dropna(inplace=True)

# Calculate Cumulative Strategy Returns (Equity Curve)
signals_with_position['Cumulative_Strategy_Returns'] = (1 + signals_with_position['Strategy_Returns']).cumprod() - 1
signals_with_position['Cumulative_Market_Returns'] = (1 + signals_with_position['Market_Returns']).cumprod() - 1

# 20. Backtesting Performance Chart (Equity Curve)
print("Generating Backtesting Performance Chart (Equity Curve)...")

fig = go.Figure()
fig.add_trace(go.Scatter(x=signals_with_position.index, y=signals_with_position['Cumulative_Strategy_Returns'], mode='lines', name='Strategy Returns', line=dict(color='yellow')))
fig.add_trace(go.Scatter(x=signals_with_position.index, y=signals_with_position['Cumulative_Market_Returns'], mode='lines', name='Market Returns', line=dict(color='blue')))

fig.update_layout(
    title_text='Strategy vs Market Cumulative Returns (Equity Curve)',
    xaxis_title="Date",
    yaxis_title="Cumulative Returns",
    hovermode="x unified",
    template="plotly_dark"
)
fig.show()

# --- Additional Performance Metrics ---

# Total Profit/Loss
final_strategy_return = signals_with_position['Cumulative_Strategy_Returns'].iloc[-1]
final_market_return = signals_with_position['Cumulative_Market_Returns'].iloc[-1]

# Strategy Return % (already calculated as final_strategy_return)

# Total Trades
# From the trade_df created in STEP 5
total_trades = len(trade_df)

# Win Rate
wins = 0
losses = 0
for i in range(0, len(trade_df), 2): # Iterate through buy-sell pairs
    if i + 1 < len(trade_df): # Ensure there's a corresponding sell trade
        buy_price = trade_df.iloc[i]['Price']
        sell_price = trade_df.iloc[i+1]['Price']
        if sell_price > buy_price:
            wins += 1
        else:
            losses += 1

win_rate = (wins / (wins + losses)) * 100 if (wins + losses) > 0 else 0

# Risk Ratio (Profit Factor: Total Gross Profit / Total Gross Loss)
# This is a simplification; a more accurate risk ratio would consider max drawdown, VaR, etc.
# Let's calculate total profit and total loss for closed trades
total_gross_profit = 0
total_gross_loss = 0

for i in range(0, len(trade_df), 2):
    if i + 1 < len(trade_df):
        buy_price = trade_df.iloc[i]['Price']
        sell_price = trade_df.iloc[i+1]['Price']
        profit_loss = sell_price - buy_price
        if profit_loss > 0:
            total_gross_profit += profit_loss
        else:
            total_gross_loss += abs(profit_loss)

risk_ratio = total_gross_profit / total_gross_loss if total_gross_loss > 0 else np.inf

print("\n--- Backtesting Results ---")
print(f"Total Trades: {total_trades}")
print(f"Strategy Return %: {final_strategy_return:.2%}")
print(f"Market Return %: {final_market_return:.2%}")
print(f"Win Rate: {win_rate:.2f}%")
print(f"Risk Ratio (Profit Factor): {risk_ratio:.2f}")

print("\n--- Backtesting Step Complete ---")


--- STEP 6: BACKTESTING ---
Generating Backtesting Performance Chart (Equity Curve)...


/tmp/ipykernel_1610/3332452513.py:19: FutureWarning:

ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


/tmp/ipykernel_1610/3332452513.py:20: FutureWarning:

ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained ass


--- Backtesting Results ---
Total Trades: 25
Strategy Return %: 28.91%
Market Return %: 7.23%
Win Rate: 33.33%
Risk Ratio (Profit Factor): 1.45

--- Backtesting Step Complete ---


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import pandas as pd # Ensure pandas is imported
import numpy as np # Ensure numpy is imported

# Set a light, colorful template as requested
import plotly.io as pio
pio.templates.default = "plotly_white"

print("\n--- STEP 7: PROFESSIONAL UI DESIGN - Creating a Comprehensive Dashboard ---")

# --- Dashboard Creation ---
# Create subplots for the dashboard layout
# The layout is designed to show key insights at a glance with interactivity.
fig = make_subplots(
    rows=5, cols=2,
    row_heights=[0.25, 0.15, 0.2, 0.2, 0.2], # Adjust heights for visual balance
    specs=[
        [{"rowspan": 1, "colspan": 2, "type": "ohlc"}, None], # Row 1: Candlestick (full width)
        [{"rowspan": 1, "colspan": 2, "type": "xy"}, None],  # Row 2: Volume (full width, using xy for bar chart)
        [{"type": "xy"}, {"type": "xy"}], # Row 3: MA Signals | Cumulative Returns
        [{"type": "xy"}, {"type": "xy"}], # Row 4: ARIMA Forecast | Daily Returns Histogram
        [{"type": "xy"}, {"type": "xy"}], # Row 5: Scatter Plot | Monthly Performance Heatmap
    ],
    vertical_spacing=0.05,
    horizontal_spacing=0.03,
    subplot_titles=(
        'Candlestick Chart',
        'Trading Volume',
        'MA Crossover Trading Signals',
        'Strategy vs Market Cumulative Returns',
        'ARIMA 30-Day Forecast',
        'Daily Returns Distribution',
        'Daily Returns vs Volume Scatter',
        'Monthly Performance Heatmap'
    )
)

# 1. Candlestick Chart (Row 1, Col 1-2)
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df['Open'],
    high=df['High'],
    low=df['Low'],
    close=df['Close'],
    name='Candlestick'
), row=1, col=1)

# 2. Trading Volume Bar Chart (Row 2, Col 1-2)
fig.add_trace(go.Bar(
    x=df.index,
    y=df['Volume'],
    name='Volume',
    marker_color='mediumpurple' # Colorful choice
), row=2, col=1)

# 3. Closing Price with MAs & Trading Signals (Row 3, Col 1)
fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Close Price', line=dict(color='gray', width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['MA_20'], mode='lines', name='20-Day MA', line=dict(color='limegreen', dash='dot')), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['MA_50'], mode='lines', name='50-Day MA', line=dict(color='firebrick', dash='dash')), row=3, col=1)

# Plot Buy Signals
buy_signals_filtered = signals[signals['Position'] == 1]
fig.add_trace(go.Scatter(
    x=buy_signals_filtered.index,
    y=df['Close'].loc[buy_signals_filtered.index],
    mode='markers',
    marker=dict(symbol='triangle-up', size=8, color='green', line=dict(width=1, color='DarkSlateGrey')),
    name='Buy Signal'
), row=3, col=1)

# Plot Sell Signals
sell_signals_filtered = signals[signals['Position'] == -1]
fig.add_trace(go.Scatter(
    x=sell_signals_filtered.index,
    y=df['Close'].loc[sell_signals_filtered.index],
    mode='markers',
    marker=dict(symbol='triangle-down', size=8, color='red', line=dict(width=1, color='DarkSlateGrey')),
    name='Sell Signal'
), row=3, col=1)

# 4. Cumulative Returns (Strategy vs Market) (Row 3, Col 2)
fig.add_trace(go.Scatter(x=signals_with_position.index, y=signals_with_position['Cumulative_Strategy_Returns'], mode='lines', name='Strategy Returns', line=dict(color='dodgerblue')), row=3, col=2)
fig.add_trace(go.Scatter(x=signals_with_position.index, y=signals_with_position['Cumulative_Market_Returns'], mode='lines', name='Market Returns', line=dict(color='darkorange')), row=3, col=2)

# 5. ARIMA Forecast (Row 4, Col 1)
fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Actual Close Price', line=dict(color='purple', width=1)), row=4, col=1)
fig.add_trace(go.Scatter(x=forecast_series.index, y=forecast_series.values, mode='lines', name='ARIMA Forecast', line=dict(color='forestgreen', dash='dash')), row=4, col=1)

# 6. Daily Returns Histogram (Row 4, Col 2)
fig.add_trace(go.Histogram(x=df['Daily_Returns'].dropna(), nbinsx=50, name='Daily Returns', marker_color='teal'), row=4, col=2)

# 7. Scatter Plot: Daily Returns vs Volume (Row 5, Col 1)
fig.add_trace(go.Scatter(x=df['Volume'], y=df['Daily_Returns'], mode='markers', name='Returns vs Volume',
                         marker=dict(color='crimson', size=3, opacity=0.6)), row=5, col=1)

# 8. Monthly Performance Heatmap (Row 5, Col 2)
fig.add_trace(go.Heatmap(
    z=monthly_performance.values,
    x=monthly_performance.columns,
    y=monthly_performance.index,
    colorscale='RdYlGn',
    colorbar_title='Return',
    name='Monthly Performance',
    hovertemplate = 'Year: %{x}<br>Month: %{y}<br>Return: %{z:.2%}<extra></extra>'
), row=5, col=2)


# Update layout for overall dashboard and add unified date range slider
fig.update_layout(
    title_text='Gland stock price analysis dashboard', # Updated title
    height=1500, # Adjust height to accommodate all charts
    hovermode="x unified",
    xaxis_rangeslider_visible=False, # Hide individual candlestick slider from layout default
    template="plotly_white"
)

# Add a unified date range slider and selector buttons to the primary time-series x-axis
# This rangeslider is applied to the x-axis of the main Candlestick chart and then linked to others.
fig.update_xaxes(
    rangeslider_visible=True, # Explicitly make rangeslider visible for this axis
    rangeselector=dict(
        buttons=list([
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=6, label="6m", step="month", stepmode="backward"),
            dict(count=1, label="YTD", step="year", stepmode="todate"),
            dict(count=1, label="1y", step="year", stepmode="backward"),
            dict(step="all")
        ])
    ),
    row=1, col=1 # Apply to the x-axis of the main Candlestick chart
)

# Link x-axes of other time-series plots to the main one
fig.update_xaxes(matches='x', row=2, col=1)
fig.update_xaxes(matches='x', row=3, col=1)
fig.update_xaxes(matches='x', row=3, col=2)
fig.update_xaxes(matches='x', row=4, col=1)

# --- Add Year-based Slicer (Dropdown) ---
years = df.index.year.unique().tolist()
years.sort()

dropdown_buttons = []
# Add an 'All Years' option
dropdown_buttons.append(dict(
    label='All Years',
    method='relayout',
    args=[{'xaxis.range': [df.index.min(), df.index.max()]}]
))

for year in years:
    # Filter data for the specific year
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'
    dropdown_buttons.append(dict(
        label=str(year),
        method='relayout',
        args=[{'xaxis.range': [start_date, end_date]}]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            type='dropdown',
            direction='down',
            x=0.01, # Position of the dropdown (left side)
            y=1.08, # Position above the chart
            showactive=True,
            active=0, # 'All Years' is default
            buttons=dropdown_buttons,
            xanchor='left',
            yanchor='top'
        )
    ]
)

# Update y-axis titles for clarity
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.update_yaxes(title_text="Price", row=3, col=1)
fig.update_yaxes(title_text="Cumulative Returns", row=3, col=2)
fig.update_yaxes(title_text="Price", row=4, col=1)
fig.update_yaxes(title_text="Frequency", row=4, col=2)
fig.update_xaxes(title_text="Volume", row=5, col=1) # Specific x-axis for scatter
fig.update_yaxes(title_text="Daily Returns", row=5, col=1) # Specific y-axis for scatter
fig.update_xaxes(title_text="Year", row=5, col=2) # Specific x-axis for heatmap
fig.update_yaxes(title_text="Month", row=5, col=2) # Specific y-axis for heatmap

fig.show()

print("\n--- Comprehensive Dashboard Created ---")



--- STEP 7: PROFESSIONAL UI DESIGN - Creating a Comprehensive Dashboard ---



--- Comprehensive Dashboard Created ---
